In [1]:
import glob
import zipfile
import pandas as pd

# 1. Find and unzip the uploaded zip file
zip_files = glob.glob('*.zip')
print("Found zip file(s):", zip_files)

with zipfile.ZipFile(zip_files[0], 'r') as zip_ref:
  zip_ref.extractall('concert_data')

# 2. Find and inspect all CSVs inside
csv_files = glob.glob('concert_data/**/*.csv', recursive=True)
print("Extracted CSV(s):", csv_files)

for f in csv_files:
  df = pd.read_csv(f)
  print(f"\n=== {f} ===")
  print("Shape:", df.shape)
  print("Columns:", list(df.columns))
  print("\nMissing Values:\n", df.isnull().sum())
  display(df.head(5))

Found zip file(s): ['archive (1).zip']
Extracted CSV(s): ['concert_data/my_file (1).csv']

=== concert_data/my_file (1).csv ===
Shape: (20, 11)
Columns: ['Rank', 'Peak', 'All Time Peak', 'Actual\xa0gross', 'Adjusted\xa0gross (in 2022 dollars)', 'Artist', 'Tour title', 'Year(s)', 'Shows', 'Average gross', 'Ref.']

Missing Values:
 Rank                                 0
Peak                                11
All Time Peak                       14
Actual gross                         0
Adjusted gross (in 2022 dollars)     0
Artist                               0
Tour title                           0
Year(s)                              0
Shows                                0
Average gross                        0
Ref.                                 0
dtype: int64


,Rank,Peak,All Time Peak,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Year(s),Shows,Average gross,Ref.
0,1,1,2,"$780,000,000","$780,000,000",Taylor Swift,The Eras Tour †,2023–2024,56,"$13,928,571",[1]
1,2,1,7[2],"$579,800,000","$579,800,000",Beyoncé,Renaissance World Tour,2023,56,"$10,353,571",[3]
2,3,1[4],2[5],"$411,000,000","$560,622,615",Madonna,Sticky & Sweet Tour ‡[4][a],2008–2009,85,"$4,835,294",[6]
3,4,2[7],10[7],"$397,300,000","$454,751,555",Pink,Beautiful Trauma World Tour,2018–2019,156,"$2,546,795",[7]
4,5,2[4],NaN,"$345,675,146","$402,844,849",Taylor Swift,Reputation Stadium Tour,2018,53,"$6,522,173",[8]


In [2]:
import re
import pandas as pd

# 1. Load the dataset
df = pd.read_csv('concert_data/my_file (1).csv')

# 2. Fix hidden non-breaking spaces (\xa0) in column names
df.columns = [
    col.replace('\xa0', ' ').strip() for col in df.columns
]

# 3. Drop the Ref. column (not needed for analysis)
df = df.drop(columns=['Ref.'])

# 4. Clean Tour title: remove footnotes like [4][a] and symbols like †, ‡, *
df['Tour title'] = (
    df['Tour title']
    .astype(str)
    .str.replace(r'\[.*?\]', '', regex=True)
    .str.replace(r'[†‡*]', '', regex=True)
    .str.strip()
)

# 5. Clean currency columns (remove $, commas, footnotes, and convert to numeric)
currency_cols = [
    'Actual gross',
    'Adjusted gross (in 2022 dollars)',
    'Average gross',
]
for col in currency_cols:
  df[col] = (
      df[col]
      .astype(str)
      .str.replace(r'\[.*?\]', '', regex=True)
      .str.replace(r'[^0-9.]', '', regex=True)
  )
  df[col] = pd.to_numeric(df[col], errors='coerce')

# 6. Clean Peak & All Time Peak (strip footnotes, keep as numeric with NaNs)
for col in ['Peak', 'All Time Peak']:
  df[col] = (
      df[col]
      .astype(str)
      .str.replace(r'\[.*?\]', '', regex=True)
      .str.replace(r'[^0-9]', '', regex=True)
  )
  df[col] = pd.to_numeric(df[col], errors='coerce')

# 7. Split Year(s) into Start Year, End Year, and Duration
years = df['Year(s)'].astype(str).str.split(r'[–-]', expand=True)
df['Start Year'] = pd.to_numeric(years[0].str.strip(), errors='coerce')
df['End Year'] = pd.to_numeric(years[1].str.strip(), errors='coerce').fillna(
    df['Start Year']
).astype(int)
df['Tour Duration (Years)'] = df['End Year'] - df['Start Year'] + 1

# 8. Data Validation: Recalculate Average Gross per Show to verify accuracy
df['Calculated Avg Gross'] = (df['Actual gross'] / df['Shows']).round(0)
df['Inflation Impact ($)'] = (
    df['Adjusted gross (in 2022 dollars)'] - df['Actual gross']
)

# 9. Inspect the cleaned data and export for Tableau
print('Cleaned Shape:', df.shape)
display(df.head(10))

df.to_csv('cleaned_concert_tours.csv', index=False)

Cleaned Shape: (20, 15)


,Rank,Peak,All Time Peak,Actual gross,Adjusted gross (in 2022 dollars),Artist,Tour title,Year(s),Shows,Average gross,Start Year,End Year,Tour Duration (Years),Calculated Avg Gross,Inflation Impact ($)
0,1,1.0,2.0,780000000,780000000,Taylor Swift,The Eras Tour,2023–2024,56,13928571,2023,2024,2,13928571.0,0
1,2,1.0,7.0,579800000,579800000,Beyoncé,Renaissance World Tour,2023,56,10353571,2023,2023,1,10353571.0,0
2,3,1.0,2.0,411000000,560622615,Madonna,Sticky & Sweet Tour,2008–2009,85,4835294,2008,2009,2,4835294.0,149622615
3,4,2.0,10.0,397300000,454751555,Pink,Beautiful Trauma World Tour,2018–2019,156,2546795,2018,2019,2,2546795.0,57451555
4,5,2.0,NaN,345675146,402844849,Taylor Swift,Reputation Stadium Tour,2018,53,6522173,2018,2018,1,6522173.0,57169703
5,6,2.0,10.0,305158363,388978496,Madonna,The MDNA Tour,2012,88,3467709,2012,2012,1,3467709.0,83820133
6,7,2.0,NaN,280000000,381932682,Celine Dion,Taking Chances World Tour,2008–2009,131,2137405,2008,2009,2,2137405.0,101932682
7,7,NaN,NaN,257600000,257600000,Pink,Summer Carnival,2023–2024,41,6282927,2023,2024,2,6282927.0,0
8,9,NaN,NaN,256084556,312258401,Beyoncé,The Formation World Tour,2016,49,5226215,2016,2016,1,5226215.0,56173845
9,10,NaN,NaN,250400000,309141878,Taylor Swift,The 1989 World Tour,2015,85,2945882,2015,2015,1,2945882.0,58741878
